In [ ]:
# # Model Training and Evaluation

# ## Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ## Load Processed Data

df = pd.read_csv("../../data/processed/house_prices_processed.csv")
print(f"Data shape: {df.shape}")
df.head()

In [ ]:
# ## Prepare Data

# Select features
features = ['SQFT', 'BEDROOMS', 'PRICE_PER_SQFT', 'LOCATION', 'REGION', 'TITLED', 'LEASE', 'FOOTINGS']
target = 'PRICE'

X = df[features].copy()
y = df[target].copy()

# One-hot encode categorical variables
categorical_cols = ['LOCATION', 'REGION', 'TITLED']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Add numerical features
numerical_cols = ['SQFT', 'BEDROOMS', 'PRICE_PER_SQFT', 'FOOTINGS', 'LEASE']
# Ensure all features are numeric
X_final = X_encoded.astype(float)

print(f"Features shape: {X_final.shape}")
print(f"Features: {X_final.columns.tolist()}")

In [ ]:
# ## Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# ## Baseline Models

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }
    
    print(f"{name} Performance:")
    print(f"  MAE: {results[name]['MAE']:.4f}")
    print(f"  RMSE: {results[name]['RMSE']:.4f}")
    print(f"  R2 Score: {results[name]['R2']:.4f}")

In [ ]:
# ## Model Comparison

results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    results_df[metric].plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'{metric} Comparison')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ## Hyperparameter Tuning

# ### XGBoost Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("\nPerforming Grid Search for XGBoost...")
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# ### Evaluate Best Model
best_xgb = grid_search.best_estimator_
y_pred_best = best_xgb.predict(X_test)
best_r2 = r2_score(y_test, y_pred_best)

print(f"\nBest XGBoost R2 Score: {best_r2:.4f}")

In [ ]:
# ## Cross Validation

# Cross validation for best model
cv_scores = cross_val_score(best_xgb, X_final, y, cv=5, scoring='r2')
print(f"\nCross-validation R2 scores: {cv_scores}")
print(f"Mean CV R2: {np.mean(cv_scores):.4f}")
print(f"Std CV R2: {np.std(cv_scores):.4f}")

In [ ]:
# ## Feature Importance

plt.figure(figsize=(10, 8))
feature_importance = pd.Series(
    best_xgb.feature_importances_,
    index=X_final.columns
).sort_values()

feature_importance.plot(kind='barh')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# ## Model Saving

import joblib

# Save best model
joblib.dump(best_xgb, '../../artifacts/trained_models/xgboost_best.joblib')
print("Best model saved!")

# Save feature columns for inference
joblib.dump(X_final.columns.tolist(), '../../artifacts/feature_columns.joblib')
print("Feature columns saved!")

In [ ]:
# ## Test Prediction

# Create sample input
sample_input = pd.DataFrame({
    'SQFT': [1500],
    'BEDROOMS': [3],
    'PRICE_PER_SQFT': [150],
    'FOOTINGS': [1],
    'LEASE': [0],
    'LOCATION_Suburban': [1],
    'LOCATION_Urban': [0],
    'REGION_Midwest': [0],
    'REGION_Northeast': [0],
    'REGION_South': [0],
    'REGION_West': [1],
    'TITLED_Land-Home': [0],
    'TITLED_Other': [0],
    'TITLED_Vehicle': [1]
})

# Make prediction
prediction = best_xgb.predict(sample_input)[0]
print(f"\nTest Prediction:")
print(f"Input: {sample_input.iloc[0].to_dict()}")
print(f"Predicted Price: ${prediction:,.2f}")